# Control de lectura 3
## Verificación numérica de gradientes mediante diferencias finitas
**20 minutos · Individual · Notebook principal**

Compare dos gradientes candidatos con una aproximación numérica. No necesita derivar
a mano, usar PyTorch ni entrenar una neurona. La lectura previa se realiza antes del control.
Complete una sola expresión, conserve su predicción inicial y justifique con sus resultados.

**Nombre:** [ESCRIBA SU NOMBRE]

## Forma de trabajo
1. Responda P0 antes de ejecutar (3 minutos).
2. Complete `diferencia_central` (5 minutos).
3. Ejecute y observe las dos tablas (5 minutos).
4. Responda P1 a P3 (5 minutos).
5. Guarde y entregue (2 minutos).

Modifique solo nombre, respuestas y la expresión pendiente. Use `Shift + Enter`.
No borre celdas. El mensaje PENDIENTE indica que falta su implementación, no es una solución.
Si cambia la función, reejecute las celdas desde arriba. No redondee cálculos ni cambie
el punto o los pasos. La notación `1e-5` significa diez elevado a menos cinco.

P0. Antes de ejecutar: ¿qué dos cantidades se comparan al verificar un gradiente? ¿Un h menor siempre mejora la aproximación? Justifique y conserve su predicción.

**PREDICCIÓN (escrita antes de ejecutar):**

Se comparan dos cantidades: (1) el **gradiente numérico** `g_num`, obtenido por diferencias centrales evaluando únicamente la función `f` en `w+h` y `w-h`, y (2) el **gradiente candidato**, calculado analíticamente (aquí `g_A` o `g_B`). La comparación es válida precisamente porque `g_num` constituye evidencia independiente: no utiliza la fórmula de la derivada.

**No, un h menor NO siempre mejora la aproximación.** Predigo dos efectos que compiten entre sí:

- **Error de truncamiento:** disminuye al reducir h (la diferencia central es de orden $O(h^2)$).
- **Error de redondeo:** *aumenta* al reducir h, porque $f(w+h)$ y $f(w-h)$ se vuelven casi iguales y al restarlos se cancelan las cifras significativas; el resultado se divide luego entre un $2h$ minúsculo, lo que amplifica el ruido.

Por tanto espero que exista un h intermedio óptimo. Predigo en concreto que con $h = 10^{-16}$ el resultado será inservible: $2.0 + 10^{-16}$ no es representable como distinto de $2.0$ en un float con 53 bits de mantisa, así que `w+h` y `w-h` quedarán almacenados como el mismo número, el numerador dará exactamente 0 y `g_num` valdrá 0 **por un problema de representación, no porque la derivada real sea cero**.

## Función y candidatos proporcionados
$$f(w)=w^3-2w,\qquad g_A(w)=3w^2-2,\qquad g_B(w)=3w-2.$$
Compruebe en **w = 2** usando diferencias centrales con los tres pasos proporcionados.
El objetivo es obtener evidencia independiente de los candidatos: no llame a sus
funciones para calcular la aproximación numérica ni devuelva un valor fijo.

In [1]:
import csv
import math
import sys
from pathlib import Path

# Funciones y configuración proporcionadas. No modificar.
def funcion(w):
    return w**3 - 2*w

def gradiente_A(w):
    return 3*w**2 - 2

def gradiente_B(w):
    return 3*w - 2

w = 2.0
pasos = (1e-1, 1e-5, 1e-16)
assert sys.float_info.radix == 2 and sys.float_info.mant_dig == 53, "Se requiere float binario de doble precisión. Consulte al docente."
print("Entorno preparado: float binario de doble precisión, sin bibliotecas externas.")

Entorno preparado: float binario de doble precisión, sin bibliotecas externas.


## Complete su función
Sustituya `...` por una expresión que use la función recibida `f`, el punto `w` y el paso `h`.
Debe devolver la aproximación por diferencias centrales estudiada en la lectura.
Conserve el resto de esta celda: la guarda solo permite mostrar PENDIENTE si falta su expresión.

In [2]:
def diferencia_central(f, w, h):
    aproximacion = (f(w + h) - f(w - h)) / (2 * h)  # diferencia central
    if aproximacion is Ellipsis:
        return None
    return aproximacion

## Obtenga la evidencia
El código imprime `g_num`, los candidatos y sus discrepancias **absolutas** con la
aproximación: `abs(g_num - g_candidato)`. Estas columnas permiten comparar este caso
de escala fija; no son una regla universal de validación. La segunda tabla muestra
los puntos perturbados y una comparación de igualdad de los valores almacenados.

In [3]:
# Código proporcionado. Usa su función; no la sustituye por una solución.
resultados = []
print(f"{'h':>10} {'g_num':>18} {'g_A':>8} {'g_B':>8} {'discrep_A':>14} {'discrep_B':>14}")
for h in pasos:
    g = diferencia_central(funcion, w, h)
    if g is None:
        print(f"{h:10.0e}  PENDIENTE: complete diferencia_central y reejecute desde arriba.")
        continue
    if not isinstance(g, (float, int)) or not math.isfinite(g):
        raise ValueError("La función debe devolver un número real finito para estos casos.")
    a, b = gradiente_A(w), gradiente_B(w)
    fila = dict(h=h, g_num=g, g_A=a, g_B=b,
                discrep_A=abs(g-a), discrep_B=abs(g-b),
                w_mas_h=w+h, w_menos_h=w-h, puntos_iguales=(w+h == w-h))
    resultados.append(fila)
    print(f"{h:10.0e} {g:18.12g} {a:8.4g} {b:8.4g} {abs(g-a):14.6e} {abs(g-b):14.6e}")

print("\nPuntos almacenados (17 cifras significativas; no se redondean los cálculos):")
print(f"{'h':>10} {'w + h':>23} {'w - h':>23} {'¿iguales?':>11}")
for h in pasos:
    print(f"{h:10.0e} {w+h:23.17g} {w-h:23.17g} {str(w+h == w-h):>11}")

if len(resultados) == len(pasos):
    destino = Path("resultados_control_3.csv")
    with destino.open("w", newline="", encoding="utf-8") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=list(resultados[0]))
        escritor.writeheader()
        escritor.writerows(resultados)
    print("\nResultados guardados. Su existencia no certifica que la función o las respuestas sean correctas.")
else:
    print("\nACTIVIDAD INCOMPLETA. No se ha generado un CSV nuevo; un archivo anterior no acredita este intento.")

         h              g_num      g_A      g_B      discrep_A      discrep_B
     1e-01              10.01       10        4   1.000000e-02   6.010000e+00
     1e-05      10.0000000002       10        4   1.987388e-10   6.000000e+00
     1e-16                  0       10        4   1.000000e+01   4.000000e+00

Puntos almacenados (17 cifras significativas; no se redondean los cálculos):
         h                   w + h                   w - h   ¿iguales?
     1e-01      2.1000000000000001      1.8999999999999999       False
     1e-05      2.0000100000000001      1.9999899999999999       False
     1e-16                       2                       2        True

Resultados guardados. Su existencia no certifica que la función o las respuestas sean correctas.


P1. ¿Qué candidato respalda una fila fiable? Cite h, la aproximación y ambas discrepancias para justificar su elección.

La evidencia respalda al **candidato A**, $g_A(w) = 3w^2 - 2$.

La fila fiable es la de $h = 10^{-5}$, el paso intermedio: es lo bastante pequeño para que el error de truncamiento sea despreciable y lo bastante grande para que `w+h` y `w-h` sigan siendo números distintos en memoria. La segunda tabla lo confirma: `w+h = 2.0000100000000001`, `w-h = 1.9999899999999999`, ¿iguales? `False`.

| | valor |
|---|---|
| h | 1e-05 |
| g_num | 10.0000000002 |
| g_A(2) | 10.0 → **discrep_A = 1.987388e-10** |
| g_B(2) | 4.0 → **discrep_B = 6.000000e+00** |

La discrepancia con A es del orden de $10^{-10}$, compatible con el error residual esperado del método; la discrepancia con B es de 6.0, magnitud comparable al propio valor del gradiente, es decir, un desacuerdo total.

La fila $h = 10^{-1}$ apunta en la misma dirección (`g_num = 10.01`, `discrep_A = 1.0e-02` frente a `discrep_B = 6.01`), aunque su discrepancia con A es mayor porque allí domina el error de truncamiento. La fila $h = 10^{-16}$ **no** sirve como evidencia (ver P2).

*Nota:* esta pequeña discrepancia apoya la **compatibilidad local** de `g_A` en $w=2$; no constituye una prueba de igualdad exacta ni una tolerancia universal (ver P3).

P2. Compare el menor h con el intermedio: aproximación, w + h y w - h. Explique la causa del cambio y contraste con P0.

| h | g_num | w + h | w - h | ¿iguales? |
|---|---|---|---|---|
| 1e-05 | 10.0000000002 | 2.0000100000000001 | 1.9999899999999999 | False |
| 1e-16 | **0.0** | 2 | 2 | **True** |

**Causa.** El paso más pequeño da un resultado *peor*, no mejor. Un float de doble precisión tiene 53 bits de mantisa, así que cerca de $2.0$ el espaciado entre números representables consecutivos (el ULP) es de orden $2^{-51} pprox 4.4	imes10^{-16}$, **mayor que** $h = 10^{-16}$. Al calcular $2.0 + 10^{-16}$ el resultado se redondea de vuelta al mismo float $2.0$, y lo mismo ocurre con $2.0 - 10^{-16}$. Los dos puntos, matemáticamente distintos, quedan almacenados como **el mismo valor**: la columna ¿iguales? lo confirma con `True`. Entonces $f(w+h) - f(w-h) = 0$ exactamente, y $g_{num} = 0 / (2	imes10^{-16}) = 0$.

Ese 0 es un **artefacto de la representación en punto flotante**, no una afirmación sobre la derivada: la derivada real en $w=2$ vale 10. Obsérvese además que con $h = 10^{-16}$ la discrepancia menor corresponde a **B** (4.0 frente a 10.0 de A), de modo que tomar esa fila como evidencia llevaría a elegir el candidato equivocado. Una fila numéricamente degenerada puede aparentar que "favorece" al candidato incorrecto.

**Contraste con P0.** Mi predicción se confirma. Anticipé que un h menor no siempre mejora, por la competencia entre error de truncamiento y error de redondeo, y que $10^{-16}$ colapsaría a numerador cero por representación. Los tres pasos muestran ese comportamiento **no monótono**: $10^{-1}$ tiene discrepancia $10^{-2}$ (domina el truncamiento), $10^{-5}$ alcanza $2	imes10^{-10}$ (mejor equilibrio) y $10^{-16}$ se degrada por completo (domina la cancelación). El óptimo está en un h **intermedio**, no en el más pequeño posible.

P3. ¿Coincidir en w = 2 demuestra corrección para cualquier entrada? Justifique y proponga otra comprobación, sin programarla.

**No.** Coincidir en un solo punto no demuestra corrección general: es evidencia **local**, no una certificación de la implementación. La comprobación solo indica que ambas funciones toman valores compatibles en $w=2$, no que sean la misma función.

El propio ejercicio lo ilustra bien. Un candidato erróneo puede coincidir por casualidad en puntos aislados: $g_A(w) = 3w^2-2$ y $g_B(w) = 3w-2$ son iguales exactamente donde $3w^2 = 3w$, es decir en $w=0$ y $w=1$. Si el control se hubiera planteado en $w=1$, ambos candidatos habrían dado 1 y la prueba no habría distinguido nada, pese a que B es incorrecto. Pasar en un punto es condición **necesaria, no suficiente**.

**Otra comprobación propuesta (sin programarla).** Repetir la verificación en varios puntos elegidos deliberadamente distintos entre sí, incluyendo al menos un valor negativo y evitando los puntos de coincidencia accidental: por ejemplo $w = -3$, $w = 0.5$ y $w = 4$. En cada punto se usaría de nuevo un h intermedio (del orden de $10^{-5}$, escalado al tamaño de $w$ para que `w+h` y `w-h` sigan siendo distintos en memoria) y se compararían ambas discrepancias. Se exigiría que el candidato mantenga una discrepancia pequeña en **todos** los puntos; basta un punto con desacuerdo claro para descartarlo. En $w=-3$, por ejemplo, $g_A$ daría 25 y $g_B$ daría $-11$, valores muy separados, de modo que el punto resulta informativo.

Complementariamente, y como indica la lectura previa, si el problema tuviera varios parámetros se perturbaría **una componente cada vez** manteniendo las demás y los datos fijos, comprobando así una componente del gradiente por vez.

## Entrega y evaluación
Reinicie el kernel, ejecute todo y guarde. Entregue este notebook con su nombre, función,
tablas y respuestas P0 a P3, junto con `resultados_control_3.csv`.
Si usa la alternativa `.py`, entregue el script con las respuestas y el CSV.
Si no termina, conserve su trabajo parcial e indique qué falta.

Se evalúan implementación (30%), evidencia (30%) e interpretación (40%). Una predicción
inicialmente incorrecta puede recibir crédito si está razonada y se contrasta honestamente.
El programa no califica sus respuestas ni verifica el orden en que las escribió.
Los resultados y las respuestas deben corresponder a esta ejecución, no a un CSV anterior.